# Cross-Context Composition Paper Robustness Audit

This notebook audits the completed Qwen-27B cross-context factor-composition experiment without loading the model or running new interventions.

[实验设置与运行说明](../../../docs/experiment_notes/10_exploratory_composition_and_gating/421ae4f554011f9d.md)


In [ ]:
# Run once. Every later code cell reports progress.
import re
import threading
import time
from IPython.core.magic import register_cell_magic
from tqdm.auto import tqdm

_SETUP_PROGRESS = tqdm(total=1, desc='Initialize progress display', unit='cell')
_ACTIVE_CELL_PROGRESS = None

def cell_stage(message):
    if _ACTIVE_CELL_PROGRESS is not None:
        _ACTIVE_CELL_PROGRESS['stage'] = str(message)
        elapsed = time.monotonic() - _ACTIVE_CELL_PROGRESS['started']
        _ACTIVE_CELL_PROGRESS['bar'].set_postfix_str(
            f"{message} | elapsed {elapsed:.0f}s", refresh=True
        )

@register_cell_magic
def cell_progress(line, cell):
    global _ACTIVE_CELL_PROGRESS
    match = re.fullmatch(r'(\d+)/(\d+)\s+(.+)', line.strip())
    current, total, label = int(match.group(1)), int(match.group(2)), match.group(3)
    started = time.monotonic()
    bar = tqdm(total=total, initial=current - 1, desc=label, unit='cell', dynamic_ncols=True)
    state = {'bar': bar, 'started': started, 'stage': 'starting'}
    _ACTIVE_CELL_PROGRESS = state
    stopped = threading.Event()
    def heartbeat():
        while not stopped.wait(1.0):
            elapsed = time.monotonic() - started
            bar.set_postfix_str(
                f"{state['stage']} | elapsed {elapsed:.0f}s", refresh=True
            )
    worker = threading.Thread(target=heartbeat, daemon=True)
    worker.start()
    succeeded = False
    try:
        result = get_ipython().run_cell(cell)
        result.raise_error()
        succeeded = True
        return result.result
    finally:
        stopped.set()
        worker.join(timeout=2)
        elapsed = time.monotonic() - started
        bar.n = current if succeeded else current - 1
        bar.set_postfix_str(
            f"{'complete' if succeeded else 'failed'} | elapsed {elapsed:.0f}s",
            refresh=True,
        )
        bar.close()
        _ACTIVE_CELL_PROGRESS = None

_SETUP_PROGRESS.update(1)
_SETUP_PROGRESS.close()
print('Progress helper ready: 11 tracked cells.')


In [ ]:
%%cell_progress 1/11 Mount Drive and locate the completed experiment
cell_stage('Mounting Google Drive')
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
PROJECT_CANDIDATES = (
    Path('/content/drive/MyDrive/VisualTorque'),
    Path('/content/drive/MyDrive/VisualTorque/VisualTorque'),
)
PROJECT_ROOT = next(
    (path for path in PROJECT_CANDIDATES if (path / 'results').is_dir()),
    None,
)
assert PROJECT_ROOT is not None, 'Could not locate VisualTorque/results on Drive.'
RESULT_ROOT = (
    PROJECT_ROOT / 'results'
    / 'qwen27b_final_crosscontext_factor_composition_v1'
)
ANALYSIS_ROOT = RESULT_ROOT / 'analysis'
AUDIT_ROOT = RESULT_ROOT / 'paper_robustness_audit'
FIGURE_ROOT = AUDIT_ROOT / 'figures'
AUDIT_ROOT.mkdir(parents=True, exist_ok=True)
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)
print('Completed experiment:', RESULT_ROOT)
print('Audit output:', AUDIT_ROOT)


In [ ]:
%%cell_progress 2/11 Import the analysis environment and lock the audit
cell_stage('Importing CPU-only analysis dependencies')
import hashlib
import itertools
import json
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

SEED = 20261102
BOOTSTRAP_REPLICATES = 20000
ALPHA = 0.05
FACTORS = ('weight', 'distance')
PRIMARY_METRICS = (
    'shared_progress',
    'shared_progress_advantage_over_random',
    'shared_vs_residual_progress',
    'route_progress_attenuation',
    'route_progress_specificity',
    'shared_slope',
    'raw_target_slope',
)
EXPECTED_POSITIVE_METRICS = (
    'shared_progress_advantage_over_random',
    'route_progress_attenuation',
    'route_progress_specificity',
)

def stable_seed(value):
    digest = hashlib.sha256(f'{SEED}:{value!r}'.encode('utf-8')).hexdigest()
    return int(digest[:16], 16) % (2**63 - 1)

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def read_jsonl(path):
    with Path(path).open('r', encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

def json_default(value):
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    raise TypeError(type(value).__name__)

def write_json_atomic(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(
        json.dumps(value, indent=2, ensure_ascii=False, default=json_default) + '\n',
        encoding='utf-8',
    )
    temporary.replace(path)

LOCK_PATH = AUDIT_ROOT / 'audit_protocol.json'
audit_identity = {
    'schema_version': 'crosscontext-composition-paper-robustness-1.0',
    'source_result': str(RESULT_ROOT),
    'independent_unit': 'family_id',
    'factors': list(FACTORS),
    'primary_metrics': list(PRIMARY_METRICS),
    'checks': [
        'exact_sign_flip', 'leave_one_family_out', 'mapping_subgroups',
        'physical_side_subgroups', 'factor_change_sign_subgroups',
        'prior_scene_exclusion', 'target_outcome_variation',
    ],
    'bootstrap_replicates': BOOTSTRAP_REPLICATES,
    'seed': SEED,
}
if LOCK_PATH.is_file():
    assert read_json(LOCK_PATH)['identity'] == audit_identity
else:
    write_json_atomic(LOCK_PATH, {
        'identity': audit_identity,
        'locked_at_utc': datetime.now(timezone.utc).isoformat(),
    })
print('Audit protocol locked.')


In [ ]:
%%cell_progress 3/11 Load and verify the paper-facing inputs
cell_stage('Loading records, family estimands, manifests, and frozen decisions')

required = {
    'completion': RESULT_ROOT / 'RUN_COMPLETE.json',
    'protocol': RESULT_ROOT / 'frozen_protocol.json',
    'manifest': RESULT_ROOT / 'confirmation_manifest.jsonl',
    'manifest_lock': RESULT_ROOT / 'confirmation_manifest_lock.json',
    'records': ANALYSIS_ROOT / 'factor_equivalence_records.csv',
    'families': ANALYSIS_ROOT / 'family_estimands.csv',
    'tests': ANALYSIS_ROOT / 'factor_equivalence_tests.csv',
    'decision': ANALYSIS_ROOT / 'factor_equivalence_decision.json',
    'audit': ANALYSIS_ROOT / 'factor_equivalence_audit.json',
}
for name, path in required.items():
    assert path.is_file(), f'Missing {name}: {path}'

completion = read_json(required['completion'])
original_decision = read_json(required['decision'])
original_audit = read_json(required['audit'])
manifest_rows = read_jsonl(required['manifest'])
record_frame = pd.read_csv(required['records'])
family_frame = pd.read_csv(required['families'])
test_frame = pd.read_csv(required['tests'])

assert completion['primary_decision'] == original_decision['primary_decision']
assert len(family_frame) == completion['families'] == len(manifest_rows)
assert set(family_frame['family_id']) == {
    row['family_id'] for row in manifest_rows
}
assert original_audit['physics_violations'] == 0
assert original_audit['nonfinite_count'] == 0
assert original_audit['selection_uses_intervention_outputs'] is False
assert family_frame.groupby('factor').size().to_dict() == {
    'distance': 10, 'weight': 10,
}
assert set(PRIMARY_METRICS).issubset(family_frame.columns)

print('Original decision:', original_decision['primary_decision'])
print('Families / target cells / records:',
      len(family_frame), original_audit['active_target_cells'], len(record_frame))
display(test_frame[test_frame['metric'].isin(PRIMARY_METRICS)].round(6))


In [ ]:
%%cell_progress 4/11 Attach mapping, side, overlap, and support metadata
cell_stage('Reconstructing family-level support without using intervention effects')

prior_manifest_path = (
    PROJECT_ROOT / 'results' / 'qwen27b_graded_product_law_v1'
    / 'graded_product_manifest.jsonl'
)
prior_scene_ids = set()
if prior_manifest_path.is_file():
    for row in read_jsonl(prior_manifest_path):
        prior_scene_ids.update(
            str(role['scene_id']) for role in row.get('roles', {}).values()
        )

metadata_rows = []
for row in manifest_rows:
    active_roles = [
        f'source_{key}_{suffix}'
        for key in ('s1', 's2', 's3') for suffix in ('base', 'changed')
    ]
    active_roles.extend(
        f'target_{key}_{suffix}'
        for key in row['active_target_keys'] for suffix in ('base', 'hybrid')
    )
    active_scene_ids = {
        str(row['roles'][role_name]['scene_id']) for role_name in active_roles
    }
    active_prior_overlap = len(active_scene_ids & prior_scene_ids)
    metadata_rows.append({
        'family_id': row['family_id'],
        'factor': row['factor'],
        'changed_side': row['changed_side'],
        'delta_sign': int(row['delta_sign']),
        'mapping_variant': ('forward', 'reverse')[
            int(row['counterbalance_phase']) % 2
        ],
        'active_target_count': len(row['active_target_keys']),
        'target_outcomes_vary': bool(
            row.get('screen_active_target_outcomes_vary', False)
        ),
        'active_prior_overlap_count': active_prior_overlap,
        'prior_overlap_group': 'overlap' if active_prior_overlap else 'fresh',
    })
metadata_frame = pd.DataFrame(metadata_rows)
assert not metadata_frame.duplicated('family_id').any()
analysis_frame = family_frame.merge(
    metadata_frame, on=['family_id', 'factor'], how='left', validate='one_to_one'
)
assert not analysis_frame.isna().any().any()

support = analysis_frame.groupby([
    'factor', 'changed_side', 'delta_sign', 'mapping_variant',
    'prior_overlap_group',
]).size().rename('n').reset_index()
support.to_csv(AUDIT_ROOT / 'support_cells.csv', index=False)
metadata_frame.to_csv(AUDIT_ROOT / 'family_metadata.csv', index=False)
display(support)
print('Strictly fresh families by factor:',
      analysis_frame[analysis_frame['active_prior_overlap_count'] == 0]
      .groupby('factor').size().to_dict())


In [ ]:
%%cell_progress 5/11 Reproduce main effects with exact small-sample tests
cell_stage('Using independent-family means and exact sign flips')

def bootstrap_mean(values, key):
    values = np.asarray(values, dtype=float)
    generator = np.random.default_rng(stable_seed(key))
    draws = generator.integers(
        0, len(values), size=(BOOTSTRAP_REPLICATES, len(values))
    )
    estimates = values[draws].mean(axis=1)
    return (
        float(values.mean()),
        float(np.quantile(estimates, ALPHA / 2)),
        float(np.quantile(estimates, 1 - ALPHA / 2)),
    )

def exact_positive_sign_flip_p(values):
    values = np.asarray(values, dtype=float)
    observed = float(values.mean())
    count = 0
    total = 2 ** len(values)
    for bits in itertools.product((-1.0, 1.0), repeat=len(values)):
        estimate = float(np.mean(values * np.asarray(bits)))
        count += int(estimate >= observed - 1e-12)
    return float(count / total)

main_rows = []
for factor in FACTORS:
    current = analysis_frame[analysis_frame['factor'] == factor]
    for metric in PRIMARY_METRICS:
        estimate, low, high = bootstrap_mean(
            current[metric], ('main', factor, metric)
        )
        main_rows.append({
            'factor': factor, 'metric': metric, 'n_families': len(current),
            'estimate': estimate, 'ci_low': low, 'ci_high': high,
            'exact_one_sided_sign_flip_p': exact_positive_sign_flip_p(
                current[metric]
            ),
        })
main_frame = pd.DataFrame(main_rows)
main_frame.to_csv(AUDIT_ROOT / 'main_exact_sensitivity.csv', index=False)
display(main_frame.round(6))


In [ ]:
%%cell_progress 6/11 Run leave-one-family-out influence checks
cell_stage('Deleting each independent family once')

loo_rows = []
for factor in FACTORS:
    current = analysis_frame[analysis_frame['factor'] == factor]
    for metric in PRIMARY_METRICS:
        for omitted in current['family_id']:
            retained = current[current['family_id'] != omitted]
            loo_rows.append({
                'factor': factor, 'metric': metric,
                'omitted_family_id': omitted,
                'estimate': float(retained[metric].mean()),
            })
loo_frame = pd.DataFrame(loo_rows)
loo_frame.to_csv(AUDIT_ROOT / 'leave_one_family_out.csv', index=False)
loo_summary = loo_frame.groupby(['factor', 'metric'])['estimate'].agg(
    minimum='min', maximum='max', mean='mean',
    all_positive=lambda values: bool((values > 0).all()),
    fraction_positive=lambda values: float((values > 0).mean()),
).reset_index()
loo_summary.to_csv(AUDIT_ROOT / 'leave_one_family_out_summary.csv', index=False)
display(loo_summary.round(6))


In [ ]:
%%cell_progress 7/11 Audit answer mapping, physical side, and change direction
cell_stage('Estimating descriptive family-level subgroup effects')

GROUP_VARIABLES = (
    'mapping_variant', 'changed_side', 'delta_sign', 'active_target_count',
)
subgroup_rows = []
for factor in FACTORS:
    factor_frame = analysis_frame[analysis_frame['factor'] == factor]
    for group_variable in GROUP_VARIABLES:
        for group_value, current in factor_frame.groupby(group_variable):
            for metric in PRIMARY_METRICS:
                estimate, low, high = bootstrap_mean(
                    current[metric],
                    ('subgroup', factor, group_variable, str(group_value), metric),
                )
                subgroup_rows.append({
                    'factor': factor, 'group_variable': group_variable,
                    'group_value': str(group_value), 'metric': metric,
                    'n_families': len(current), 'estimate': estimate,
                    'ci_low': low, 'ci_high': high,
                    'inferential_status': (
                        'descriptive_only' if len(current) < 4 else 'limited_ci'
                    ),
                })
subgroup_frame = pd.DataFrame(subgroup_rows)
subgroup_frame.to_csv(AUDIT_ROOT / 'subgroup_sensitivity.csv', index=False)
display(subgroup_frame[
    subgroup_frame['metric'].isin(EXPECTED_POSITIVE_METRICS)
].round(6))


In [ ]:
%%cell_progress 8/11 Exclude prior support and tighten target variation
cell_stage('Repeating estimates on stricter paper-facing subsets')

subset_definitions = {
    'all_families': np.ones(len(analysis_frame), dtype=bool),
    'no_prior_scene_overlap': (
        analysis_frame['active_prior_overlap_count'].to_numpy() == 0
    ),
    'target_outcomes_vary': analysis_frame['target_outcomes_vary'].to_numpy(),
    'no_overlap_and_target_outcomes_vary': (
        (analysis_frame['active_prior_overlap_count'].to_numpy() == 0)
        & analysis_frame['target_outcomes_vary'].to_numpy()
    ),
    'three_active_targets': (
        analysis_frame['active_target_count'].to_numpy() == 3
    ),
}
subset_rows = []
for subset_name, mask in subset_definitions.items():
    subset = analysis_frame.loc[mask]
    for factor in FACTORS:
        current = subset[subset['factor'] == factor]
        for metric in PRIMARY_METRICS:
            if len(current) < 2:
                subset_rows.append({
                    'subset': subset_name, 'factor': factor, 'metric': metric,
                    'n_families': len(current), 'estimate': np.nan,
                    'ci_low': np.nan, 'ci_high': np.nan,
                    'status': 'insufficient_support',
                })
                continue
            estimate, low, high = bootstrap_mean(
                current[metric], ('subset', subset_name, factor, metric)
            )
            subset_rows.append({
                'subset': subset_name, 'factor': factor, 'metric': metric,
                'n_families': len(current), 'estimate': estimate,
                'ci_low': low, 'ci_high': high,
                'status': 'descriptive_only' if len(current) < 4 else 'limited_ci',
            })
subset_frame = pd.DataFrame(subset_rows)
subset_frame.to_csv(AUDIT_ROOT / 'strict_subset_sensitivity.csv', index=False)
display(subset_frame[
    subset_frame['metric'].isin(EXPECTED_POSITIVE_METRICS)
].round(6))


In [ ]:
%%cell_progress 9/11 Classify robustness without changing the frozen decision
cell_stage('Separating supported claims from subgroup limitations')

def original_gate(factor, name):
    return bool(original_decision['factor_decisions'][factor]['gates'][name])

def loo_all_positive(factor, metric):
    return bool(loo_summary[
        (loo_summary['factor'] == factor) & (loo_summary['metric'] == metric)
    ]['all_positive'].iloc[0])

def subgroup_positive(factor, metric, variable):
    current = subgroup_frame[
        (subgroup_frame['factor'] == factor)
        & (subgroup_frame['metric'] == metric)
        & (subgroup_frame['group_variable'] == variable)
    ]
    return bool(
        len(current) >= 2
        and (current['n_families'] >= 2).all()
        and (current['estimate'] > 0).all()
    )

def subset_positive(factor, metric, subset_name, minimum_n=4):
    current = subset_frame[
        (subset_frame['factor'] == factor)
        & (subset_frame['metric'] == metric)
        & (subset_frame['subset'] == subset_name)
    ].iloc[0]
    return bool(
        int(current['n_families']) >= minimum_n
        and np.isfinite(current['estimate'])
        and float(current['estimate']) > 0
    )

flags = {
    'weight': {
        'original_shared_beats_random': original_gate(
            'weight', 'shared_beats_equal_norm_random'
        ),
        'original_route_mediation': bool(
            original_decision['factor_decisions']['weight'][
                'route_mediation_confirmed'
            ]
        ),
        'leave_one_out_shared_over_random_positive': loo_all_positive(
            'weight', 'shared_progress_advantage_over_random'
        ),
        'leave_one_out_route_attenuation_positive': loo_all_positive(
            'weight', 'route_progress_attenuation'
        ),
        'leave_one_out_route_specificity_positive': loo_all_positive(
            'weight', 'route_progress_specificity'
        ),
        'both_mapping_means_route_positive': subgroup_positive(
            'weight', 'route_progress_specificity', 'mapping_variant'
        ),
        'both_side_means_route_positive': subgroup_positive(
            'weight', 'route_progress_specificity', 'changed_side'
        ),
        'fresh_subset_shared_over_random_positive': subset_positive(
            'weight', 'shared_progress_advantage_over_random',
            'no_prior_scene_overlap',
        ),
        'fresh_subset_route_specificity_positive': subset_positive(
            'weight', 'route_progress_specificity',
            'no_prior_scene_overlap',
        ),
    },
    'distance': {
        'strict_factor_equivalence_confirmed': bool(
            original_decision['factor_decisions']['distance'][
                'factor_equivalence_confirmed'
            ]
        ),
        'route_mediation_confirmed': bool(
            original_decision['factor_decisions']['distance'][
                'route_mediation_confirmed'
            ]
        ),
        'both_mapping_means_shared_over_random_positive': subgroup_positive(
            'distance', 'shared_progress_advantage_over_random',
            'mapping_variant',
        ),
        'both_side_means_shared_over_random_positive': subgroup_positive(
            'distance', 'shared_progress_advantage_over_random', 'changed_side'
        ),
    },
}

weight_core = all(flags['weight'][key] for key in (
    'original_shared_beats_random', 'original_route_mediation',
    'leave_one_out_shared_over_random_positive',
    'leave_one_out_route_attenuation_positive',
    'leave_one_out_route_specificity_positive',
))
weight_broad = all(flags['weight'].values())
if weight_core and weight_broad:
    robustness_decision = 'weight_mixed_state_reuse_and_route_broadly_robust'
elif weight_core:
    robustness_decision = 'weight_core_robust_with_subgroup_scope_limits'
else:
    robustness_decision = 'weight_effect_sensitive_to_family_influence'

decision = {
    'schema_version': 'crosscontext-composition-paper-robustness-decision-1.0',
    'original_frozen_decision': original_decision['primary_decision'],
    'robustness_decision': robustness_decision,
    'flags': flags,
    'claim_boundary': (
        'This audit can support robustness of partial weight reuse and its '
        'frozen route. It cannot convert the failed factor-equivalence gate '
        'into evidence for a pure variable or a stable W x D product law.'
    ),
}
write_json_atomic(AUDIT_ROOT / 'robustness_decision.json', decision)
print('Robustness decision:', robustness_decision)
display(pd.DataFrame([
    {'factor': factor, 'check': key, 'passed': value}
    for factor, values in flags.items() for key, value in values.items()
]))


In [ ]:
%%cell_progress 10/11 Save the paper-facing robustness figure
cell_stage('Rendering family influence, subgroups, and strict subsets')

colors = {'weight': '#2563EB', 'distance': '#D97706'}
metric_labels = {
    'shared_progress_advantage_over_random': 'Shared − random',
    'route_progress_attenuation': 'Route attenuation',
    'route_progress_specificity': 'Route specificity',
}
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

for factor in FACTORS:
    current = analysis_frame[analysis_frame['factor'] == factor].sort_values(
        'shared_progress_advantage_over_random'
    )
    axes[0, 0].scatter(
        np.arange(len(current)), current['shared_progress_advantage_over_random'],
        label=factor, color=colors[factor], s=55,
    )
axes[0, 0].axhline(0, color='black', linewidth=1)
axes[0, 0].set_title('A. Independent-family shared-minus-random effects')
axes[0, 0].set_xlabel('Family rank within factor')
axes[0, 0].set_ylabel('Progress difference')
axes[0, 0].legend(frameon=False)

plot_loo = loo_summary[loo_summary['metric'].isin(
    metric_labels
)].copy()
positions = np.arange(len(metric_labels))
for offset, factor in zip((-0.12, 0.12), FACTORS):
    current = plot_loo[plot_loo['factor'] == factor].set_index('metric')
    lows = [current.loc[metric, 'minimum'] for metric in metric_labels]
    highs = [current.loc[metric, 'maximum'] for metric in metric_labels]
    means = [(low + high) / 2 for low, high in zip(lows, highs)]
    errors = np.vstack([
        np.asarray(means) - np.asarray(lows),
        np.asarray(highs) - np.asarray(means),
    ])
    axes[0, 1].errorbar(
        positions + offset, means, yerr=errors, fmt='o', capsize=4,
        label=factor, color=colors[factor],
    )
axes[0, 1].axhline(0, color='black', linewidth=1)
axes[0, 1].set_xticks(positions, [metric_labels[m] for m in metric_labels], rotation=18)
axes[0, 1].set_title('B. Range after deleting any one family')
axes[0, 1].set_ylabel('Leave-one-out estimate')
axes[0, 1].legend(frameon=False)

mapping_plot = subgroup_frame[
    (subgroup_frame['metric'] == 'route_progress_specificity')
    & (subgroup_frame['group_variable'] == 'mapping_variant')
].copy()
labels = []
values = []
lows = []
highs = []
bar_colors = []
for factor in FACTORS:
    for mapping in ('forward', 'reverse'):
        current = mapping_plot[
            (mapping_plot['factor'] == factor)
            & (mapping_plot['group_value'] == mapping)
        ]
        if current.empty:
            continue
        row = current.iloc[0]
        labels.append(f'{factor}\n{mapping}\nn={int(row.n_families)}')
        values.append(row.estimate)
        lows.append(row.estimate - row.ci_low)
        highs.append(row.ci_high - row.estimate)
        bar_colors.append(colors[factor])
axes[1, 0].bar(
    np.arange(len(values)), values, color=bar_colors,
    yerr=np.vstack([lows, highs]), capsize=4,
)
axes[1, 0].axhline(0, color='black', linewidth=1)
axes[1, 0].set_xticks(np.arange(len(values)), labels)
axes[1, 0].set_title('C. Route specificity by answer mapping')
axes[1, 0].set_ylabel('Progress specificity')

strict_plot = subset_frame[
    subset_frame['metric'].isin(metric_labels)
    & subset_frame['subset'].isin(('all_families', 'no_prior_scene_overlap'))
].copy()
strict_labels = []
strict_values = []
strict_lows = []
strict_highs = []
strict_colors = []
for factor in FACTORS:
    for subset in ('all_families', 'no_prior_scene_overlap'):
        for metric in metric_labels:
            current = strict_plot[
                (strict_plot['factor'] == factor)
                & (strict_plot['subset'] == subset)
                & (strict_plot['metric'] == metric)
            ]
            if current.empty or not np.isfinite(current.iloc[0].estimate):
                continue
            row = current.iloc[0]
            strict_labels.append(
                f'{factor[0].upper()} {"all" if subset == "all_families" else "fresh"}\n'
                f'{metric_labels[metric]}\nn={int(row.n_families)}'
            )
            strict_values.append(row.estimate)
            strict_lows.append(row.estimate - row.ci_low)
            strict_highs.append(row.ci_high - row.estimate)
            strict_colors.append(colors[factor])
axes[1, 1].bar(
    np.arange(len(strict_values)), strict_values, color=strict_colors,
    yerr=np.vstack([strict_lows, strict_highs]), capsize=3,
)
axes[1, 1].axhline(0, color='black', linewidth=1)
axes[1, 1].set_xticks(np.arange(len(strict_values)), strict_labels, rotation=35, ha='right')
axes[1, 1].set_title('D. All support versus prior-scene-free support')
axes[1, 1].set_ylabel('Family-level effect')

fig.suptitle('Qwen-27B Cross-Context Composition Robustness', fontsize=16)
fig.tight_layout(rect=(0, 0, 1, 0.97))
figure_path = FIGURE_ROOT / 'crosscontext_composition_robustness.png'
fig.savefig(figure_path, dpi=180, bbox_inches='tight')
plt.show()
print('Figure:', figure_path)


In [ ]:
%%cell_progress 11/11 Write the paper-facing audit report
cell_stage('Saving concise claims, tables, and completion metadata')

def main_interval(factor, metric):
    row = main_frame[
        (main_frame['factor'] == factor) & (main_frame['metric'] == metric)
    ].iloc[0]
    return f"{row.estimate:.5f} [{row.ci_low:.5f}, {row.ci_high:.5f}]"

def loo_range(factor, metric):
    row = loo_summary[
        (loo_summary['factor'] == factor) & (loo_summary['metric'] == metric)
    ].iloc[0]
    return f"{row.minimum:.5f} to {row.maximum:.5f}"

lines = [
    '# Cross-Context Composition Paper Robustness Audit', '',
    f"**Frozen original decision:** {original_decision['primary_decision']}", '',
    f"**Robustness decision:** {robustness_decision}", '',
    '## Support', '',
    f"- Independent families: {len(analysis_frame)}",
    f"- Weight / distance families: {dict(analysis_frame.groupby('factor').size())}",
    f"- Families with no prior graded-composition scene overlap: "
    f"{dict(analysis_frame[analysis_frame.active_prior_overlap_count == 0].groupby('factor').size())}",
    '', '## Weight', '',
    f"- Shared over random: {main_interval('weight', 'shared_progress_advantage_over_random')}",
    f"- Route attenuation: {main_interval('weight', 'route_progress_attenuation')}",
    f"- Route specificity: {main_interval('weight', 'route_progress_specificity')}",
    f"- Leave-one-family-out shared-over-random range: {loo_range('weight', 'shared_progress_advantage_over_random')}",
    f"- Leave-one-family-out route-specificity range: {loo_range('weight', 'route_progress_specificity')}",
    '', '## Distance', '',
    f"- Shared over random: {main_interval('distance', 'shared_progress_advantage_over_random')}",
    f"- Route attenuation: {main_interval('distance', 'route_progress_attenuation')}",
    f"- Route specificity: {main_interval('distance', 'route_progress_specificity')}",
    '', '## Interpretation', '',
    (
        'This audit evaluates whether the partial weight-reuse and route result '
        'depends on a single family, answer mapping, physical side, or reuse of '
        'earlier graded-composition scenes. It does not alter the frozen failure '
        'of strict factor equivalence. A robust positive weight result therefore '
        'supports a causally useful but mixed internal signal, not a pure weight '
        'variable or a stable internal multiplication law.'
    ), '',
    '## Machine-readable checks', '',
]
for factor, values in flags.items():
    lines.append(f'### {factor.title()}')
    lines.append('')
    for name, value in values.items():
        lines.append(f'- {name}: {value}')
    lines.append('')

report_path = AUDIT_ROOT / 'CROSSCONTEXT_COMPOSITION_ROBUSTNESS_REPORT.md'
report_path.write_text('\n'.join(lines) + '\n', encoding='utf-8')
completion_document = {
    'schema_version': 'crosscontext-composition-paper-robustness-complete-1.0',
    'robustness_decision': robustness_decision,
    'families': int(len(analysis_frame)),
    'records': int(len(record_frame)),
    'report': str(report_path),
    'figure': str(figure_path),
    'completed_at_utc': datetime.now(timezone.utc).isoformat(),
}
write_json_atomic(AUDIT_ROOT / 'RUN_COMPLETE.json', completion_document)
print('Robustness decision:', robustness_decision)
print('Report:', report_path)
print('Run complete:', AUDIT_ROOT / 'RUN_COMPLETE.json')
